[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.1 MB/s eta 0:00:00


In [3]:
import torch

In [40]:
# ✏️ YOUR IMPLEMENTATION HERE

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    # pass  # maintain beams, expand, prune, return best
    beams = [(0.0, [start_token])]
    completed = []

    # extending
    for step in range(max_len):
      candidates = []
      for score, seq in beams:
          if seq[-1] == eos_token:
              # end, without rollout
              completed.append((score, seq))
              continue
          # continue rollout, and insert the beam_width top candidates
          log_probs = log_prob_fn(torch.tensor(seq))
          top_scores, top_tokens = torch.topk(log_probs, beam_width)
          for token_score, token in zip(top_scores, top_tokens):
              new_score = score + token_score.item()
              new_seq = seq + [token.item()]
              # print(f"{new_score} {new_seq}")
              candidates.append((new_score, new_seq))
              # for score_, seq_ in candidates:
              #   print(f"candidates: {score_} {seq_}")

          # beams = top beam_width candidates by score
          candidates.sort(key=lambda x: x[0], reverse=True)
          # print("-------candidates\n")
          # for score_, seq_ in candidates:
          #       print(f"\tcandidates: {score_} {seq_}")
          beams = candidates[:beam_width]
    all_seqs = completed + beams
    all_seqs.sort(key=lambda x: x[0], reverse=True) # sort all beams
    return all_seqs[0][1]


In [41]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

Sequence: [0, 1, 2, 3, 4]


In [42]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')


🧪 Testing: Beam Search Decoding (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Returns list starting with start_token (2.8ms)
  ✅ [2/4] Greedy path (beam=1) (1.2ms)
  ✅ [3/4] Beam finds better path than greedy (0.6ms)
  ✅ [4/4] Stops at eos (4.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (9.3ms total)
  Progress saved. Run status() to see your dashboard.

